# 🔥 Train-LLM: Sistema de Predicción de Incendios - Guatemala

## Notebook 01: Descarga y Exploración de Datos

**Objetivo**: Descargar datos históricos de NASA FIRMS (2014-2024) y realizar exploración inicial.

**Fases del PLAN.md cubiertas**:
- ✅ Fase 0: Configuración inicial
- ✅ Fase 1: Adquisición de datos NASA FIRMS
- ✅ Fase 1.3: Exploración y visualización preliminar

---

**Antes de empezar**:
1. Registrate en NASA FIRMS: https://firms.modaps.eosdis.nasa.gov/api/area/
2. Copia tu MAP_KEY cuando llegue al email
3. Pegala en la celda de configuración abajo

---

## 📦 FASE 0: Instalación de Dependencias

In [ ]:
# Instalar dependencias necesarias
!pip install -q requests pandas numpy matplotlib seaborn plotly folium tqdm

print("✓ Dependencias instaladas")

In [ ]:
# Imports
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import folium
from folium.plugins import HeatMap
from datetime import datetime
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficas
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Librerías importadas")

In [ ]:
# Crear estructura de directorios en Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Crear carpetas del proyecto
!mkdir -p /content/drive/MyDrive/train_llm_fires/data/raw
!mkdir -p /content/drive/MyDrive/train_llm_fires/data/processed
!mkdir -p /content/drive/MyDrive/train_llm_fires/visualizations
!mkdir -p /content/drive/MyDrive/train_llm_fires/models

print("✓ Estructura de directorios creada en Google Drive")

## 🔑 FASE 0.4: Configuración de API Keys

In [ ]:
# ⚠️ IMPORTANTE: Reemplaza con tu MAP_KEY de NASA FIRMS
# Registrate aquí: https://firms.modaps.eosdis.nasa.gov/api/area/

FIRMS_MAP_KEY = "TU_MAP_KEY_AQUI"  # ⬅️ REEMPLAZAR

# Configuración del proyecto
PROJECT_NAME = "train-llm-guatemala-fires"
COUNTRY_CODE = "GTM"  # Guatemala
START_DATE = "2014-01-01"
END_DATE = "2024-12-31"

# Bounding box Guatemala (lat_min, lat_max, lon_min, lon_max)
GUATEMALA_BBOX = {
    'lat_min': 13.74,
    'lat_max': 17.82,
    'lon_min': -92.23,
    'lon_max': -88.23
}

print(f"✓ Configuración cargada: {PROJECT_NAME}")
print(f"  País: Guatemala ({COUNTRY_CODE})")
print(f"  Período: {START_DATE} a {END_DATE}")
print(f"  Área: {GUATEMALA_BBOX}")

## 📡 FASE 1.1: Descarga de Datos NASA FIRMS

NASA FIRMS (Fire Information for Resource Management System) proporciona detecciones de puntos calientes en tiempo casi real desde satélites MODIS y VIIRS.

**Sensores disponibles**:
- `VIIRS_SNPP_NRT`: Mejor resolución espacial (375m)
- `MODIS_NRT`: Mayor cobertura temporal
- `VIIRS_NOAA20_NRT`: VIIRS en satélite NOAA-20

Vamos a descargar los 3 y combinarlos.

In [ ]:
def download_firms_data(map_key, country='GTM', sensor='VIIRS_SNPP_NRT', days=10):
    """
    Descarga datos FIRMS desde la API de NASA.
    
    Parámetros:
    - map_key: API key de FIRMS
    - country: Código de país (GTM para Guatemala)
    - sensor: VIIRS_SNPP_NRT, MODIS_NRT, VIIRS_NOAA20_NRT
    - days: Número de años (máximo 10)
    
    Retorna:
    - DataFrame de pandas con las detecciones
    """
    # Construir URL de la API
    # Nota: days=10 descarga últimos 10 años desde START_DATE
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/country/{country}/{days}/{START_DATE}"
    
    print(f"\n📡 Descargando datos {sensor}...")
    print(f"   URL: {url[:80]}...")
    
    try:
        response = requests.get(url, timeout=300)
        
        if response.status_code == 200:
            # Guardar CSV temporalmente
            temp_file = f'/content/{sensor}_{country}_temp.csv'
            with open(temp_file, 'w') as f:
                f.write(response.text)
            
            # Leer como DataFrame
            df = pd.read_csv(temp_file)
            
            # Agregar columna de sensor
            df['sensor'] = sensor
            
            print(f"   ✓ Descargado: {len(df):,} detecciones")
            print(f"   ✓ Período: {df['acq_date'].min()} a {df['acq_date'].max()}")
            
            return df
        else:
            print(f"   ✗ Error {response.status_code}: {response.text[:200]}")
            return None
            
    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        return None

print("✓ Función download_firms_data() definida")

In [ ]:
# Descargar datos de los 3 sensores
print("🚀 Iniciando descarga de datos NASA FIRMS...")
print("⏳ Esto puede tomar 2-5 minutos dependiendo de la conexión\n")

# VIIRS SNPP (mejor resolución)
df_viirs_snpp = download_firms_data(FIRMS_MAP_KEY, sensor='VIIRS_SNPP_NRT')

# MODIS (mayor cobertura histórica)
df_modis = download_firms_data(FIRMS_MAP_KEY, sensor='MODIS_NRT')

# VIIRS NOAA-20
df_viirs_noaa = download_firms_data(FIRMS_MAP_KEY, sensor='VIIRS_NOAA20_NRT')

print("\n" + "="*60)
print("✓ DESCARGA COMPLETADA")
print("="*60)

In [ ]:
# Combinar datasets
print("🔗 Combinando datasets...\n")

dataframes = [df for df in [df_viirs_snpp, df_modis, df_viirs_noaa] if df is not None]

if dataframes:
    df_combined = pd.concat(dataframes, ignore_index=True)
    
    print(f"✓ Datasets combinados:")
    print(f"  Total de detecciones: {len(df_combined):,}")
    print(f"\n  Desglose por sensor:")
    print(df_combined['sensor'].value_counts())
    
    # Guardar en Google Drive
    output_path = '/content/drive/MyDrive/train_llm_fires/data/raw/nasa_firms_guatemala_2014_2024_combined.csv'
    df_combined.to_csv(output_path, index=False)
    print(f"\n✓ Guardado en: {output_path}")
    
else:
    print("✗ No se pudo descargar ningún dataset. Verifica tu MAP_KEY.")

## 🔍 FASE 1.2: Exploración Inicial de Datos

In [ ]:
# Información general del dataset
print("📊 INFORMACIÓN DEL DATASET\n")
print(f"Dimensiones: {df_combined.shape[0]:,} filas × {df_combined.shape[1]} columnas")
print(f"\nColumnas disponibles:")
for i, col in enumerate(df_combined.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nPrimeras 5 filas:")
df_combined.head()

In [ ]:
# Estadísticas descriptivas
print("📈 ESTADÍSTICAS DESCRIPTIVAS\n")

# Variables numéricas clave
numeric_cols = ['latitude', 'longitude', 'brightness', 'bright_t31', 'frp', 'confidence']
df_combined[numeric_cols].describe()

In [ ]:
# Verificar valores faltantes
print("🔍 ANÁLISIS DE VALORES FALTANTES\n")

missing = df_combined.isnull().sum()
missing_pct = (missing / len(df_combined)) * 100

missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores Faltantes': missing.values,
    'Porcentaje': missing_pct.values
}).sort_values('Valores Faltantes', ascending=False)

print(missing_df[missing_df['Valores Faltantes'] > 0])

if missing_df['Valores Faltantes'].sum() == 0:
    print("\n✓ No hay valores faltantes!")

In [ ]:
# Crear columna datetime
df_combined['datetime'] = pd.to_datetime(
    df_combined['acq_date'] + ' ' + df_combined['acq_time'].astype(str).str.zfill(4),
    format='%Y-%m-%d %H%M'
)

# Extraer features temporales
df_combined['year'] = df_combined['datetime'].dt.year
df_combined['month'] = df_combined['datetime'].dt.month
df_combined['day'] = df_combined['datetime'].dt.day
df_combined['hour'] = df_combined['datetime'].dt.hour
df_combined['dayofweek'] = df_combined['datetime'].dt.dayofweek

print("✓ Features temporales creados")
df_combined[['datetime', 'year', 'month', 'day', 'hour']].head()

## 📊 FASE 1.3: Visualización Exploratoria

In [ ]:
# 1. Serie temporal: Incendios por año
print("📈 Generando visualizaciones...\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Incendios por año
yearly_fires = df_combined.groupby('year').size()
axes[0, 0].bar(yearly_fires.index, yearly_fires.values, color='orangered', alpha=0.7)
axes[0, 0].set_title('Incendios Detectados por Año (2014-2024)', fontsize=14, weight='bold')
axes[0, 0].set_xlabel('Año')
axes[0, 0].set_ylabel('Número de Detecciones')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Incendios por mes (estacionalidad)
monthly_fires = df_combined.groupby('month').size()
months = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
axes[0, 1].bar(range(1, 13), monthly_fires.values, color='tomato', alpha=0.7)
axes[0, 1].set_title('Estacionalidad: Incendios por Mes', fontsize=14, weight='bold')
axes[0, 1].set_xlabel('Mes')
axes[0, 1].set_ylabel('Total de Detecciones (2014-2024)')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].set_xticklabels(months, rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Distribución de confianza
axes[1, 0].hist(df_combined['confidence'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribución de Confianza de Detecciones', fontsize=14, weight='bold')
axes[1, 0].set_xlabel('Confianza (%)')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].axvline(df_combined['confidence'].mean(), color='red', linestyle='--', label=f'Media: {df_combined["confidence"].mean():.1f}%')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Fire Radiative Power (FRP)
axes[1, 1].hist(df_combined['frp'].dropna(), bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Distribución de Fire Radiative Power', fontsize=14, weight='bold')
axes[1, 1].set_xlabel('FRP (MW)')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].set_xlim(0, 500)  # Limitar para mejor visualización
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/train_llm_fires/visualizations/exploratory_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráficas guardadas en visualizations/")

In [ ]:
# Mapa de calor interactivo con Folium
print("🗺️ Generando mapa de calor interactivo...\n")

# Tomar muestra de 10,000 puntos (para performance)
sample_size = min(10000, len(df_combined))
df_sample = df_combined.sample(sample_size, random_state=42)

# Crear mapa centrado en Guatemala
center_lat = (GUATEMALA_BBOX['lat_min'] + GUATEMALA_BBOX['lat_max']) / 2
center_lon = (GUATEMALA_BBOX['lon_min'] + GUATEMALA_BBOX['lon_max']) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=7,
    tiles='OpenStreetMap'
)

# Preparar datos para heatmap
heat_data = [[row['latitude'], row['longitude']] for _, row in df_sample.iterrows()]

# Agregar capa de calor
HeatMap(
    heat_data,
    radius=10,
    blur=15,
    max_zoom=13,
    gradient={
        0.0: 'blue',
        0.3: 'lime',
        0.5: 'yellow',
        0.7: 'orange',
        1.0: 'red'
    }
).add_to(m)

# Guardar mapa
map_path = '/content/drive/MyDrive/train_llm_fires/visualizations/heatmap_guatemala.html'
m.save(map_path)

print(f"✓ Mapa de calor guardado en: {map_path}")
print(f"  Muestra: {sample_size:,} puntos de {len(df_combined):,} totales")

# Mostrar mapa en Colab
m

## 📊 FASE 1.4: Estadísticas Clave

In [ ]:
# Generar reporte de estadísticas
print("📋 REPORTE DE ESTADÍSTICAS\n")
print("="*60)

# 1. Totales
print("\n1️⃣ TOTALES")
print(f"   Total de detecciones: {len(df_combined):,}")
print(f"   Período: {df_combined['acq_date'].min()} a {df_combined['acq_date'].max()}")
print(f"   Años cubiertos: {df_combined['year'].nunique()}")

# 2. Por sensor
print("\n2️⃣ DISTRIBUCIÓN POR SENSOR")
for sensor, count in df_combined['sensor'].value_counts().items():
    pct = (count / len(df_combined)) * 100
    print(f"   {sensor:20s}: {count:8,} ({pct:5.1f}%)")

# 3. Por año
print("\n3️⃣ DISTRIBUCIÓN POR AÑO")
yearly_stats = df_combined.groupby('year').agg({
    'latitude': 'count',
    'frp': 'mean',
    'confidence': 'mean'
}).round(2)
yearly_stats.columns = ['Detecciones', 'FRP Promedio', 'Confianza Promedio']
print(yearly_stats)

# 4. Meses pico
print("\n4️⃣ MESES CON MÁS INCENDIOS (Top 5)")
monthly_totals = df_combined.groupby('month').size().sort_values(ascending=False)
for month, count in monthly_totals.head(5).items():
    month_name = months[month-1]
    pct = (count / len(df_combined)) * 100
    print(f"   {month_name:10s}: {count:8,} ({pct:5.1f}%)")

# 5. Rangos de variables
print("\n5️⃣ RANGOS DE VARIABLES")
print(f"   Brightness (K):    {df_combined['brightness'].min():.1f} - {df_combined['brightness'].max():.1f}")
print(f"   FRP (MW):          {df_combined['frp'].min():.1f} - {df_combined['frp'].max():.1f}")
print(f"   Confianza (%):     {df_combined['confidence'].min():.1f} - {df_combined['confidence'].max():.1f}")

# 6. Detecciones de alta confianza
high_conf = df_combined[df_combined['confidence'] >= 80]
print("\n6️⃣ DETECCIONES DE ALTA CONFIANZA (≥80%)")
print(f"   Total: {len(high_conf):,} ({len(high_conf)/len(df_combined)*100:.1f}%)")
print(f"   FRP promedio: {high_conf['frp'].mean():.2f} MW")

print("\n" + "="*60)

## 💾 FASE 1.5: Guardar Dataset Limpio

In [ ]:
# Filtrar datos dentro del bounding box de Guatemala
df_guatemala = df_combined[
    (df_combined['latitude'] >= GUATEMALA_BBOX['lat_min']) &
    (df_combined['latitude'] <= GUATEMALA_BBOX['lat_max']) &
    (df_combined['longitude'] >= GUATEMALA_BBOX['lon_min']) &
    (df_combined['longitude'] <= GUATEMALA_BBOX['lon_max'])
].copy()

print(f"📍 Filtrado geográfico:")
print(f"   Antes: {len(df_combined):,} detecciones")
print(f"   Después: {len(df_guatemala):,} detecciones")
print(f"   Descartadas: {len(df_combined) - len(df_guatemala):,} ({(len(df_combined) - len(df_guatemala))/len(df_combined)*100:.1f}%)")

In [ ]:
# Filtrar detecciones de baja confianza
MIN_CONFIDENCE = 50  # Umbral mínimo de confianza

df_filtered = df_guatemala[df_guatemala['confidence'] >= MIN_CONFIDENCE].copy()

print(f"🎯 Filtrado por confianza (≥{MIN_CONFIDENCE}%):")
print(f"   Antes: {len(df_guatemala):,} detecciones")
print(f"   Después: {len(df_filtered):,} detecciones")
print(f"   Descartadas: {len(df_guatemala) - len(df_filtered):,} ({(len(df_guatemala) - len(df_filtered))/len(df_guatemala)*100:.1f}%)")

In [ ]:
# Remover duplicados
df_clean = df_filtered.drop_duplicates(
    subset=['latitude', 'longitude', 'acq_date', 'acq_time']
).copy()

print(f"🔄 Eliminación de duplicados:")
print(f"   Antes: {len(df_filtered):,} detecciones")
print(f"   Después: {len(df_clean):,} detecciones")
print(f"   Duplicados eliminados: {len(df_filtered) - len(df_clean):,}")

In [ ]:
# Guardar dataset limpio
output_clean = '/content/drive/MyDrive/train_llm_fires/data/processed/fires_cleaned.csv'
df_clean.to_csv(output_clean, index=False)

print(f"\n💾 Dataset limpio guardado:")
print(f"   Ruta: {output_clean}")
print(f"   Tamaño: {len(df_clean):,} detecciones")
print(f"   Columnas: {len(df_clean.columns)}")
print(f"   Memoria: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n✅ FASE 1 COMPLETADA: Datos descargados, explorados y limpiados")

---

## ✅ RESUMEN DE FASE 1

**Completado**:
- ✅ Descarga de datos NASA FIRMS (VIIRS + MODIS)
- ✅ Exploración estadística del dataset
- ✅ Visualizaciones (serie temporal, estacionalidad, mapas)
- ✅ Limpieza de datos (filtros geográficos y de confianza)
- ✅ Guardado de dataset procesado

**Archivos generados**:
- `data/raw/nasa_firms_guatemala_2014_2024_combined.csv` (datos crudos)
- `data/processed/fires_cleaned.csv` (datos limpios)
- `visualizations/exploratory_analysis.png` (gráficas)
- `visualizations/heatmap_guatemala.html` (mapa interactivo)

**Próximos pasos (PLAN.md)**:
- 📊 **Notebook 02**: Análisis Estadístico (correlaciones, tests)
- 🤖 **Notebook 03**: Fine-tuning Qwen3-VL con Unsloth
- 🔮 **Notebook 04**: Predicción temporal 2026 (Prophet)
- 🚀 **Notebook 05**: Integración API + Dashboard

---

**¿Todo funcionó?** Si ves errores, revisa:
1. Tu MAP_KEY de FIRMS está correcta
2. Google Drive está montado
3. Conexión a internet estable

**Siguiente**: Ejecutar **Notebook 02** para análisis estadístico completo.